In [4]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

In [5]:
# Load Data
data = pd.read_csv("../data/preprocessed_data.csv")
test = pd.read_csv("../data/preprocessed_test.csv")

In [6]:

# Selected features and target
selected_columns = [
    "CryoSleep", "Age", "VIP", "RoomService", "FoodCourt",
    "ShoppingMall", "Spa", "VRDeck", "Cabin_num", "Side",
    "CabinMidFlag", "HomePlanetEarth", "HomePlanetEuropa",
    "HomePlanetMars", "Destination55 Cancri e",
    "DestinationPSO J318.5-22", "DestinationTRAPPIST-1e",
    "Group1", "Group2", "Group3", "Group4", "Group5",
    "Group6", "Group7", "DeckA", "DeckB", "DeckC",
    "DeckD", "DeckE", "DeckF", "DeckG",
    "FamilyMembersCabinCount", "avg_spending_family"
]

target_column = "Transported"

X = data[selected_columns]
y = data[target_column]

# Split into train and holdout
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

# Optional: scale numeric features 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)


In [7]:

# Base estimator
base_estimator = DecisionTreeClassifier(random_state=42)

# Define AdaBoost model
adb_model = AdaBoostClassifier(
    estimator=base_estimator,  # <-- changed from base_estimator to estimator
    random_state=42
)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "estimator__max_depth": [1, 2, 3]  # note the double underscore
}

# Grid search with 5-fold CV
grid_search = GridSearchCV(
    estimator=adb_model,
    param_grid=param_grid,
    scoring='f1',       # good for imbalanced data
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Fit grid search
grid_search.fit(X_train_scaled, y_train)

# Best hyperparameters
print("Best Parameters:", grid_search.best_params_)

# Best model
best_adb = grid_search.best_estimator_


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters: {'estimator__max_depth': 2, 'learning_rate': 0.2, 'n_estimators': 500}


In [8]:
# Predict on holdout
y_pred = best_adb.predict(X_holdout_scaled)

print(confusion_matrix(y_holdout, y_pred))
print(classification_report(y_holdout, y_pred))

[[340  92]
 [ 81 357]]
              precision    recall  f1-score   support

           0       0.81      0.79      0.80       432
           1       0.80      0.82      0.80       438

    accuracy                           0.80       870
   macro avg       0.80      0.80      0.80       870
weighted avg       0.80      0.80      0.80       870



In [9]:
# Convert to DataFrame (keep same column names if you have them)
holdout_df = pd.DataFrame(X_holdout_scaled, columns=X_holdout.columns)

# Add Actual and Pred columns
holdout_df["Actual"] = y_holdout
holdout_df["Pred"] = y_pred

# False positives and negatives
false_positive = holdout_df[(holdout_df["Pred"] == 1) & (holdout_df["Actual"] == 0)]
false_negative = holdout_df[(holdout_df["Pred"] == 0) & (holdout_df["Actual"] == 1)]

false_positive.describe(), false_negative.describe()

(       CryoSleep        Age        VIP  RoomService  FoodCourt  ShoppingMall  \
 count  22.000000  22.000000  22.000000    22.000000  22.000000     22.000000   
 mean    0.772765   0.284983   0.149704    -0.334911  -0.201269      0.243844   
 std     0.951412   1.152073   1.421981     0.002260   0.248804      1.551073   
 min    -0.745163  -1.304623  -0.153463    -0.335393  -0.281427     -0.304376   
 25%    -0.223376  -0.520143  -0.153463    -0.335393  -0.281427     -0.304376   
 50%     1.341988   0.055143  -0.153463    -0.335393  -0.281427     -0.304376   
 75%     1.341988   0.839623  -0.153463    -0.335393  -0.281427     -0.304376   
 max     1.341988   2.809541   6.516219    -0.324794   0.782640      6.326621   
 
              Spa     VRDeck  Cabin_num       Side  ...      DeckB  \
 count  22.000000  22.000000  22.000000  22.000000  ...  22.000000   
 mean   -0.262302  -0.262987   0.296972  -0.006775  ...   0.485793   
 std     0.023237   0.010683   1.235213   1.023556  ...   1

# AdaBoost

In [10]:
test_selected = test[selected_columns]
test_selected_scaled = scaler.transform(test_selected)

In [11]:
test_preds = best_adb.predict(test_selected_scaled)
test_preds = ["True" if pred == 1 else "False" for pred in test_preds]

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_preds
})

In [13]:
submission.to_csv("../output/adaboost/submission.csv", index=False)